Model 2: Few-shot LLM (max 7-8B) + RAG (if
applicable)

LLM = Large Language Model (model lingvistic de mari dimensiuni).

Este un model neuronal de tip Transformer, antrenat pe volume mari de text, care poate înțelege instrucțiuni și genera răspunsuri în limbaj natural.

Exemple: Llama și Qwen.

În proiectul meu, îl voi folosi pentru clasificarea documentelor juridice în cele 21 de categorii EUROVOC.

Ce înseamnă Few-shot LLM? 
Few-shot prompting nu modifică parametrii modelului; Greutățile rămân neschimbate, folosim instrucțiuni și exemple în prompt, nu avem epoci de antrenare, modelul este ghidat doar pentru cererea curentă.(deci opus fata de modelul 1, encoderul, unde l-am antrenat, modelul invata prin actualizarea greutatilor, etc.)

Ce înseamnă maximum 7–8B?
Litera B vine de la billion (miliard).

Un model 7B are aproximativ 7 miliarde de parametri, adică valori numerice învățate în timpul antrenării.

Profesorul impune această limită pentru a restrânge dimensiunea modelului folosit.

Important: 8B nu reprezintă numărul de documente, tokeni sau exemple oferite modelului.

Ce înseamnă Few-shot Learning?
În NLP, few-shot înseamnă că îi arătăm modelului câteva exemple de rezolvare a sarcinii, direct în prompt, înainte de a-i cere să rezolve un caz nou.


Exemplu simplificat de prompt few-shot:

Instrucțiune: Clasifică documentele în categorii EUROVOC. Un document poate avea mai multe categorii.


Exemple oferite modelului

Document: „Reglementări privind importul și exportul mărfurilor...”

Etichetă: comerț

Document: „Norme pentru cultivarea și comercializarea cerealelor...”

Etichete: agricultură, comerț

-> Document nou:
„Reguli privind transportul internațional de mărfuri...”
LLM generează: transport, comerț

Ce este RAG și ce înseamnă „if applicable”?
RAG = Retrieval-Augmented Generation, adică generare asistată de recuperarea informațiilor.

Pe scurt, înainte ca LLM-ul să răspundă, un sistem caută informații relevante într-o colecție de documente și i le introduce în prompt.

În cazul meu, RAG ar putea fi aplicabil dacă vreau ca modelul să primească exemple juridice asemănătoare documentului pe care trebuie să îl clasifice.

Cum voi rezolva Modelul 2?

Propun să compar două variante, dacă resursele permit:
- Few-shot cu exemple fixe: alegem câteva exemple reprezentative exclusiv din train și le folosim pentru toate documentele evaluate.
- Few-shot cu exemple recuperate: căutăm documente similare în train și folosim etichetele lor ca exemple relevante în prompt. Aceasta este o variantă de retrieval-augmented in-context learning, apropiată de RAG.

CONCLUZIE

Pentru Modelul 2 voi utiliza un Large Language Model de maximum 8 miliarde de parametri și voi aborda clasificarea prin few-shot prompting. Spre deosebire de fine-tuning, această metodă nu actualizează greutățile modelului, ci îi oferă câteva exemple etichetate direct în context.

Voi analiza și posibilitatea utilizării RAG pentru recuperarea unor exemple juridice similare din setul de antrenare, astfel încât clasificarea să beneficieze de context relevant.

Scopul este să compar performanța unui LLM folosit fără antrenare suplimentară cu encoderul Legal-XLM-Longformer, adaptat anterior prin fine-tuning.

Celula 1 – Încărcarea LLM-ului

Am ales Qwen/Qwen3-4B-Instruct-2507, deoarece este un model mai recent, multilingv, optimizat pentru respectarea instrucțiunilor și generarea de răspunsuri.

Cuantizarea la 4 biți este o metodă prin care reducem memoria ocupată de un model neuronal, reprezentând greutățile sale cu mai puțini biți.

Modelul Qwen are aproximativ 4 miliarde de parametri (greutăți). Fiecare parametru este o valoare numerică pe care modelul a învățat-o în timpul antrenării.

Pentru a stoca aceste valori, avem nevoie de memorie.

Ideea este că modelul păstrează o aproximare a valorii originale, nu neapărat valoarea exactă.

Astfel economisim memorie, dar introducem o mică eroare de aproximare care poate afecta predicțiile.

In [1]:

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

# Cuantizare la 4 biti pentru GPU-ul de 8 GB
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.eval()


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Model incarcat: Qwen/Qwen3-4B-Instruct-2507
Parametri: 4,022,468,096


AttributeError: 'Qwen3ForCausalLM' object has no attribute 'hf_device_map'

In [2]:
print("Model incarcat:", MODEL_NAME)
print("Parametri:", f"{model.num_parameters():,}")

# Verificam dispozitivul pe care se afla modelul
print("Dispozitiv:", model.device)

# Verificam daca modelul este cuantizat
print("Cuantizare 4-bit:", getattr(model, "is_loaded_in_4bit", False))

# Memoria GPU utilizata
print(
    "Memorie GPU alocata:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "Memorie GPU totala:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB"
)

Model incarcat: Qwen/Qwen3-4B-Instruct-2507
Parametri: 4,022,468,096
Dispozitiv: cuda:0
Cuantizare 4-bit: True
Memorie GPU alocata: 2.66 GB
Memorie GPU totala: 7.96 GB


Celula 2 – Pregătirea exemplelor MultiEURLEX

Încărcăm datele și selectăm 3 exemple din train, fără să folosim validation sau test ca exemple pentru LLM. 

Ce fac: alegem trei documente etichetate pentru a-i arăta LLM-ului cum trebuie rezolvată clasificarea multi-label. Nu antrenăm modelul.

Acestea sunt momentan exemple fixe, pentru un prim experiment few-shot. Ulterior putem verifica dacă o selecție mai reprezentativă sau recuperarea exemplelor similare (RAG) îmbunătățește rezultatele pe validation.

In [3]:

import json
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../Data/processed")

train_df = pd.read_parquet(DATA_DIR / "train.parquet")
val_df = pd.read_parquet(DATA_DIR / "validation.parquet")

with open(DATA_DIR / "id2label.json", encoding="utf-8") as f:
    id2label = json.load(f)

with open(
    DATA_DIR.parent / "eurovoc_descriptors.json",
    encoding="utf-8"
) as f:
    descriptors = json.load(f)

# Cateva exemple reproductibile, exclusiv din train
few_shot_examples = train_df.sample(
    n=3,
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))

print("\n=== EXEMPLE FEW-SHOT ===")
for _, row in few_shot_examples.iterrows():
    categories = [
        descriptors[str(label)]["ro"]
        for label in row["labels"]
    ]

    print("\nText:", row["text"][:250], "...")
    print("Categorii:", categories)


Train: 15920
Validation: 5000

=== EXEMPLE FEW-SHOT ===

Text: REGULAMENTUL (CE) NR. 209/2008 AL COMISIEI
din 6 martie 2008
privind autorizarea unei noi utilizări a Saccharomyces cerevisiae (Biosaf Sc 47) ca aditiv pentru hrana animalelor
(Text cu relevanță pentru SEE)
COMISIA COMUNITĂȚILOR EUROPENE,
având în ve ...
Categorii: ['probleme sociale', 'comerţ', 'produse agroalimentare', 'agricultură, silvicultură şi pescuit']

Text: 32005R1775
L 289/1
JURNALUL OFÍCIAL AL UNIUNII EUROPENE
REGULAMENTUL (CE) NR. 1775/2005 AL PARLAMENTULUI EUROPEAN ȘI AL CONSILIULUI
din 28 septembrie 2005
privind condițiile de acces la rețelele pentru transportul gazelor naturale
(Text cu relevanță  ...
Categorii: ['afaceri şi concurenţă', 'energie', 'transport', 'comerţ', 'UNIUNEA EUROPEANĂ']

Text: 32005D0842
L 312/67
JURNALUL OFÍCIAL AL UNIUNII EUROPENE
DECIZIA COMISIEI
din 28 noiembrie 2005
privind aplicarea articolului 86 alineatul (2) din Tratatul CE la ajutorul de stat, sub forma compensației pentru obl

Acum testez dacă Qwen poate clasifica un document nou, fără antrenare.

Limitez temporar textul la 2.000 de caractere pentru prima verificare. Aceasta nu este încă evaluarea finală; vom stabili lungimea documentelor pe validation.

Celula 3 – Prima clasificare few-shot

In [5]:
# Cele 21 de categorii EUROVOC
label_ids = sorted({
    str(label)
    for labels in train_df["labels"]
    for label in labels
})
categories = [descriptors[label]["ro"] for label in label_ids]

# Construim exemplele few-shot din train
examples_text = ""

for _, row in few_shot_examples.iterrows():
    labels = [descriptors[str(x)]["ro"] for x in row["labels"]]

    examples_text += (
        f"\nDocument:\n{row['text'][:1000]}\n"
        f"Categorii: {json.dumps(labels, ensure_ascii=False)}\n"
    )

# Primul document de validare
document = val_df.iloc[0]
true_labels = [
    descriptors[str(x)]["ro"]
    for x in document["labels"]
]

system_prompt = (
    "Ești un clasificator de documente juridice în limba română. "
    "Alege una sau mai multe categorii EUROVOC exclusiv din lista permisă. "
    "Răspunde doar cu un array JSON de categorii, fără explicații."
)

user_prompt = f"""
Categorii permise:
{json.dumps(categories, ensure_ascii=False)}

Exemple de clasificare:
{examples_text}

Clasifică următorul document:

{document['text'][:2000]}

Categorii:
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=160,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

prediction = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Categorii reale:", true_labels)
print("Predicția Qwen:", prediction)

Categorii reale: ['comerţ', 'finanţe', 'produse agroalimentare', 'agricultură, silvicultură şi pescuit']
Predicția Qwen: ["probleme sociale", "comerţ", "produse agroalimentare", "agricultură, silvicultură şi pescuit"]
